# pytorch-lightning intro

trying out lightning. the explicit pytorch loops in 2020 were fine but i kept rewriting the same boilerplate. lightning seems to factor that out.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

## simple LightningModule for MNIST

In [ ]:
class LitMLP(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.l1 = nn.Linear(28*28, 128)
        self.l2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.l1(x))
        return self.l2(x)

    def training_step(self, batch, _):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log('train_loss', loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## adding val step

In [ ]:
def validation_step(self, batch, _):
    x, y = batch
    logits = self(x)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(1) == y).float().mean()
    self.log('val_loss', loss, prog_bar=True)
    self.log('val_acc', acc, prog_bar=True)
# need to attach this to the class, will refactor

## training
following the docs more or less.

In [ ]:
tfm = transforms.Compose([transforms.ToTensor()])
train = datasets.MNIST('./data', train=True, download=True, transform=tfm)
val = datasets.MNIST('./data', train=False, download=True, transform=tfm)
tr = DataLoader(train, batch_size=128, shuffle=True, num_workers=2)
va = DataLoader(val, batch_size=256, num_workers=2)
model = LitMLP()
trainer = pl.Trainer(max_epochs=3, gpus=1 if torch.cuda.is_available() else 0)
trainer.fit(model, tr, va)

todo: try more aug.

tweaked and re-ran.

In [ ]:
# todo: try lr=3e-4

In [ ]:
# notes for next time:
#   - more epochs
#   - bigger batch

minor tweak.

In [ ]:
# add weight decay
# opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)


noticed the loss curve flattens after epoch 5. early stop wins.

In [ ]:
# linear warmup
# scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda s: min(1.0, s/100))
